# Gaussian Pulse Dispersion Simulator

This notebook simulates the propagation of a Gaussian electromagnetic wave packet through media with different dispersion properties. It visualizes how the group velocity dispersion ($dv_g/dk$) affects the pulse envelope, causing it to broaden or compress over time.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Initial parameters
v_g = 1.0        # Group velocity (assumed constant for the pulse center)
k_0 = 10.0       # Central wavenumber (k0)
w_0 = v_g * k_0  # Central angular frequency (w0)
delta_k = 0.5    # Initial spectral width (delta_k)
A_0 = 1.0        # Initial amplitude (A0)

# Spatial and temporal domain
s = np.linspace(0, 300, 800)  # Spatial axis (s)
dt = 1.5                      # Time step
times = np.arange(0, 300, dt) # Time array

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
line, = ax.plot(s, np.zeros_like(s), lw=1.5, color='blue', label='Real part of the wave')
line_env_up, = ax.plot(s, np.zeros_like(s), lw=2, color='red', linestyle='--', label='Envelope')
line_env_down, = ax.plot(s, np.zeros_like(s), lw=2, color='red', linestyle='--')

ax.set_ylim(-1.2, 1.2)
ax.set_xlim(0, 300)
ax.set_xlabel('Propagation position (s)')
ax.set_ylabel('Electromagnetic Pulse Amplitude')
ax.set_title('Evolution of a Gaussian Pulse in Dispersive Media')

# Highlight different media
ax.axvspan(0, 100, color='lightgreen', alpha=0.3, label='Medium 1: Non-dispersive')
ax.axvspan(100, 200, color='lightcoral', alpha=0.3, label='Medium 2: Positive Dispersion')
ax.axvspan(200, 300, color='lightblue', alpha=0.3, label='Medium 3: Negative Dispersion')
ax.legend(loc='upper right')

D_acc = 0.0  

def init():
    line.set_ydata(np.zeros_like(s))
    line_env_up.set_ydata(np.zeros_like(s))
    line_env_down.set_ydata(np.zeros_like(s))
    return line, line_env_up, line_env_down

# Oculta el lienzo estático inicial para que no se imprima aquí
plt.close(fig)

In [ ]:
def animate(i):
    global D_acc
    
    if i == 0:
        D_acc = 0.0
    
    t = times[i]
    
    # 1. Identify which medium the pulse center is in
    pulse_center = v_g * t
    if pulse_center < 100:
        alpha = 0.0    # dvg/dk = 0 (Non-dispersive)
    elif pulse_center < 200:
        alpha = 0.15   # dvg/dk > 0 (Positive dispersion)
    else:
        alpha = -0.15  # dvg/dk < 0 (Negative dispersion)
        
    # Accumulate the dispersion effect over time
    D_acc += alpha * dt
    
    # 2. Application of theoretical expressions
    # Term controlling the broadening: 1 + (dvg/dk)^2 * \Delta k^4 * t^2
    broadening_factor = 1.0 + (D_acc * delta_k**2)**2
    
    # The amplitude decreases to conserve the energy of the pulse
    A_t = A_0 / np.sqrt(np.sqrt(broadening_factor)) 
    
    # Calculation of the Gaussian envelope
    env_exponent = - (delta_k**2 * (s - pulse_center)**2) / (2 * broadening_factor)
    envelope = A_t * np.exp(env_exponent)
    
    # Phase calculation (including constant carrier and variable chirp)
    gouy_phase = 0.5 * np.arctan(D_acc * delta_k**2)
    chirp = - (D_acc * delta_k**4 * (s - pulse_center)**2) / (2 * broadening_factor)
    total_phase = w_0 * t - k_0 * s + gouy_phase + chirp
    
    # Combine envelope and phase to get the real perturbation
    real_pulse = envelope * np.cos(total_phase)
    
    # Update the animation curves
    line.set_ydata(real_pulse)
    line_env_up.set_ydata(envelope)
    line_env_down.set_ydata(-envelope)
    
    return line, line_env_up, line_env_down

In [ ]:
# Create the animation
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(times), interval=30, blit=True)
plt.close() # Avoid duplicate plots in the output

# Special rendering for Jupyter Notebook
HTML(ani.to_jshtml())

#### Physical Interpretation of the Simulation
This simulation models the propagation of a Gaussian wave packet using the 2nd-order Taylor expansion of $\omega(k)$. The pulse deformation depends on the group velocity dispersion term ($dv_g/dk$).

* **Non-dispersive medium ($dv_g/dk = 0$):** The signal velocity and group velocity are identical ($v_s = v_g$). All components travel at the same speed, and the wave packet moves without deformation, maintaining its original half-width and amplitude.
* **Positive dispersion medium ($dv_g/dk > 0$):** Upon entering this medium, each harmonic component travels at a different speed. The envelope progressively broadens, and its peak amplitude decreases to conserve energy.
* **Negative dispersion medium ($dv_g/dk < 0$):** By inverting the dispersive properties, the harmonic components that had advanced are now slowed down and vice versa. Physically, this compensates for the accumulated deformation, causing the wave packet to temporarily recover its maximum amplitude and minimum width before broadening again.
 